In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
movies_data = pd.read_csv('movies.csv')
ratings_data = pd.read_csv('ratings.csv')
links_data = pd.read_csv('links.csv')
tags_data = pd.read_csv('tags.csv')

In [3]:
movies_data.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings_data.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Analysis done in precheck.ipynb file

Conerting Unix timestamp measured in seconds to datetime

In [8]:
import pandas as pd

ratings = ratings_data.copy()

ratings["rating_date"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    utc=True
)

ratings = ratings.sort_values(
    ["timestamp", "userId", "movieId"]
).reset_index(drop=True)

ratings.head()

,userId,movieId,rating,timestamp,rating_date
0,429,22,4.0,828124615,1996-03-29 18:36:55+00:00
1,429,150,5.0,828124615,1996-03-29 18:36:55+00:00
2,429,161,5.0,828124615,1996-03-29 18:36:55+00:00
3,429,165,4.0,828124615,1996-03-29 18:36:55+00:00
4,429,218,4.0,828124615,1996-03-29 18:36:55+00:00


In [9]:
# Your previous cell already created and sorted "ratings".
ratings = ratings.sort_values(
    ["timestamp", "userId", "movieId"]
).reset_index(drop=True)

train_position = int(len(ratings) * 0.80)
validation_position = int(len(ratings) * 0.90)

train_cutoff = ratings["timestamp"].iloc[train_position]
validation_cutoff = ratings["timestamp"].iloc[validation_position]

if train_cutoff >= validation_cutoff:
    raise ValueError(
        "The time boundaries overlap. Choose different cutoffs."
    )

print(
    "Validation starts:",
    pd.to_datetime(train_cutoff, unit="s", utc=True)
)

print(
    "Test starts:",
    pd.to_datetime(validation_cutoff, unit="s", utc=True)
)

Validation starts: 2016-03-22 08:26:11+00:00
Test starts: 2017-06-21 07:17:30+00:00


In [10]:
train_ratings = ratings[
    ratings["timestamp"] < train_cutoff
].copy()

validation_ratings = ratings[
    (ratings["timestamp"] >= train_cutoff)
    & (ratings["timestamp"] < validation_cutoff)
].copy()

test_ratings = ratings[
    ratings["timestamp"] >= validation_cutoff
].copy()

print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(validation_ratings))
print("Test ratings:", len(test_ratings))

Training ratings: 80668
Validation ratings: 10084
Test ratings: 10084


In [13]:
split_summary = pd.DataFrame([
    {
        "split": name,
        "ratings": len(data),
        "users": data["userId"].nunique(),
        "movies": data["movieId"].nunique(),
        "first_rating": data["rating_date"].min(),
        "last_rating": data["rating_date"].max()
    }
    for name, data in [
        ("Train", train_ratings),
        ("Validation", validation_ratings),
        ("Test", test_ratings)
    ]
])

display(split_summary)

assert (
    len(train_ratings) + len(validation_ratings) + len(test_ratings)
    == len(ratings)
)

assert (
    train_ratings["timestamp"].max()
    < validation_ratings["timestamp"].min()
)

assert (
    validation_ratings["timestamp"].max()
    < test_ratings["timestamp"].min()
)

print("Split checks passed.")

,split,ratings,users,movies,first_rating,last_rating
0,Train,80668,522,7867,1996-03-29 18:36:55+00:00,2016-03-22 08:26:02+00:00
1,Validation,10084,62,3261,2016-03-22 08:26:11+00:00,2017-06-21 07:17:25+00:00
2,Test,10084,74,4428,2017-06-21 07:17:30+00:00,2018-09-24 14:27:30+00:00


Split checks passed.


In [12]:
train_users = set(train_ratings["userId"])
train_movies = set(train_ratings["movieId"])

cold_start_rows = []

for name, df in {
    "Validation": validation_ratings,
    "Test": test_ratings
}.items():

    new_user = ~df["userId"].isin(train_users)
    new_movie = ~df["movieId"].isin(train_movies)

    warm_interaction = (~new_user) & (~new_movie)

    cold_start_rows.append({
        "split": name,
        "total_interactions": len(df),
        "new_users": df.loc[
            new_user, "userId"
        ].nunique(),
        "new_movies": df.loc[
            new_movie, "movieId"
        ].nunique(),
        "new_user_interactions": int(new_user.sum()),
        "new_movie_interactions": int(new_movie.sum()),
        "either_new_interactions": int(
            (new_user | new_movie).sum()
        ),
        "warm_interactions": int(
            warm_interaction.sum()
        ),
        "warm_percentage": round(
            100 * warm_interaction.mean(), 2
        )
    })

cold_start_summary = pd.DataFrame(cold_start_rows)

display(cold_start_summary)

,split,total_interactions,new_users,new_movies,new_user_interactions,new_movie_interactions,either_new_interactions,warm_interactions,warm_percentage
0,Validation,10084,41,719,9093,1015,9374,710,7.04
1,Test,10084,54,1347,9154,2029,9483,601,5.96


In [14]:
train_users = set(train_ratings["userId"])
train_movies = set(train_ratings["movieId"])

cold_start_results = []

for name, data in [
    ("Validation", validation_ratings),
    ("Test", test_ratings)
]:
    new_user = ~data["userId"].isin(train_users)
    new_movie = ~data["movieId"].isin(train_movies)
    warm = ~new_user & ~new_movie

    cold_start_results.append({
        "split": name,
        "users_seen_in_train": data.loc[
            ~new_user, "userId"
        ].nunique(),
        "users_not_in_train": data.loc[
            new_user, "userId"
        ].nunique(),
        "movies_not_in_train": data.loc[
            new_movie, "movieId"
        ].nunique(),
        "total_ratings": len(data),
        "warm_ratings": int(warm.sum()),
        "cold_ratings": int((new_user | new_movie).sum()),
        "warm_percentage": round(100 * warm.mean(), 2)
    })

cold_start_summary = pd.DataFrame(cold_start_results)
display(cold_start_summary)

,split,users_seen_in_train,users_not_in_train,movies_not_in_train,total_ratings,warm_ratings,cold_ratings,warm_percentage
0,Validation,21,41,719,10084,710,9374,7.04
1,Test,20,54,1347,10084,601,9483,5.96


In [15]:
popular_movies = (
    train_ratings.groupby("movieId")
    .agg(
        rating_count=("rating", "size"),
        average_rating=("rating", "mean")
    )
    .reset_index()
)

popular_movies = popular_movies.merge(
    movies_data,
    on="movieId",
    how="left",
    validate="one_to_one"
)

popular_movies = popular_movies.sort_values(
    ["rating_count", "movieId"],
    ascending=[False, True]
).reset_index(drop=True)

display(
    popular_movies[
        ["movieId", "title", "genres", "rating_count", "average_rating"]
    ].head(10)
)

,movieId,title,genres,rating_count,average_rating
0,356,Forrest Gump (1994),Comedy|Drama|Romance|War,267,4.166667
1,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,264,4.189394
2,318,"Shawshank Redemption, The (1994)",Crime|Drama,256,4.423828
3,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,235,4.202128
4,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,219,4.214612
5,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,218,3.786697
6,110,Braveheart (1995),Action|Drama|War,212,4.058962
7,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,206,4.264563
8,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,200,3.970000
9,527,Schindler's List (1993),Drama|War,188,4.228723


In [16]:
def recommend_popular(seen_movie_ids=None, n=10):
    if seen_movie_ids is None:
        seen_movie_ids = []

    recommendations = popular_movies[
        ~popular_movies["movieId"].isin(seen_movie_ids)
    ]

    return recommendations[
        ["movieId", "title", "genres", "rating_count", "average_rating"]
    ].head(n).reset_index(drop=True)

In [17]:
seen_movies = [356, 296, 318]

recommendations = recommend_popular(
    seen_movie_ids=seen_movies,
    n=10
)

display(recommendations)

assert not recommendations["movieId"].isin(seen_movies).any()
print("Already-rated movies were excluded.")

,movieId,title,genres,rating_count,average_rating
0,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,235,4.202128
1,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,219,4.214612
2,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,218,3.786697
3,110,Braveheart (1995),Action|Drama|War,212,4.058962
4,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,206,4.264563
5,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,200,3.970000
6,527,Schindler's List (1993),Drama|War,188,4.228723
7,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,185,3.445946
8,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,184,3.923913
9,150,Apollo 13 (1995),Adventure|Drama|IMAX,184,3.875000


Already-rated movies were excluded.
